In [1]:
import numpy as np
from nilearn.image import load_img
import matplotlib.pyplot as plt
import scipy.stats
import pandas as pd
import scipy.io
import pickle
import os
os.chdir('/gpfs/milgram/project/chun/jk2992/socialaha/') # change to your folder path

In [2]:
def niftimask(nroi_cor, nroi_sub, directory):
    cortical = directory+'/template/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_atlas-Schaefer2018_desc-'+str(nroi_cor)+'Parcels17Networks_dseg.nii.gz'
    if nroi_sub==16: subcortical = directory+'/template/Tian2020MSA_v1.1_3T_Subcortex-Only/Tian_Subcortex_S1_3T_2009cAsym.nii.gz'
    elif nroi_sub==32: subcortical = directory+'/template/Tian2020MSA_v1.1_3T_Subcortex-Only/Tian_Subcortex_S2_3T_2009cAsym.nii.gz'
    elif nroi_sub == 50: subcortical = directory + '/template/Tian2020MSA_v1.1_3T_Subcortex-Only/Tian_Subcortex_S3_3T_2009cAsym.nii.gz'
    elif nroi_sub == 54: subcortical = directory + '/template/Tian2020MSA_v1.1_3T_Subcortex-Only/Tian_Subcortex_S4_3T_2009cAsym.nii.gz'
    mask_cor = load_img(cortical).dataobj[:]
    mask_sub = load_img(subcortical).dataobj[:]

    for i1 in range(mask_sub.shape[0]):
        for i2 in range(mask_sub.shape[1]):
            for i3 in range(mask_sub.shape[2]):
                if mask_sub[i1,i2,i3]>0:
                    mask_sub[i1,i2,i3] = mask_sub[i1,i2,i3] + nroi_cor

    id = np.where(np.multiply(mask_cor, mask_sub)>0)
    mask = mask_cor + mask_sub
    mask[id[0],id[1],id[2]] = 0
    return mask

''' setting '''
flist = {}
flist[1] = ['sub-1001', 'sub-1005', 'sub-1008', 'sub-1011', 'sub-1014', 'sub-1017', 'sub-1020', 'sub-1023', 'sub-1026', 'sub-1029', 'sub-1033', 'sub-1039']
flist[2] = ['sub-2006', 'sub-2009', 'sub-2012', 'sub-2015', 'sub-2018', 'sub-2021', 'sub-2024', 'sub-2027', 'sub-2034', 'sub-2038', 'sub-2040'] # 'sub-2030'
flist[3] = ['sub-3004', 'sub-3007', 'sub-3013', 'sub-3016', 'sub-3019', 'sub-3022', 'sub-3025', 'sub-3031', 'sub-3037', 'sub-3041'] # 'sub-3010', 'sub-3028'
tasklist = ['01','02','03','04','05','06','07','08','09','10']
# sub-2030, sub-3010, sub-3028: large head motion participants
# sub-1023 task-03: only movie watching portion was recorded
nsubj = len(flist[1])+len(flist[2])+len(flist[3])

directory = './socialaha-collab'

nroi_cor, nroi_sub = 100, 16
nroi = nroi_cor + nroi_sub
hrf = 4
mask = niftimask(nroi_cor, nroi_sub, directory)

# load the voxel time courses for the whole run from the preprocessed nifti files

In [4]:
for groupid in range(1, 3+1): # loading group 1
    allsub_ROIsum = {}
    run = np.array(pd.read_csv(directory+'/socialaha-fMRI/socialaha_groupscene.csv')['run'])
    scene = np.array(pd.read_csv(directory+'/socialaha-fMRI/socialaha_groupscene.csv')['g'+str(groupid)+'.sceneid'])

    for si, subname in enumerate(flist[groupid]):
        print('Loading data from',subname)
        roi_ts_run = {}
        for ti, task in enumerate(tasklist):
            print(subname+' task-'+task)
            # parcel mask is multiplied by each participant's, run-specific brain mask applied during preprocessing
            submask = load_img(directory+'/masks/'+subname+'/'+subname+'_combined.nii.gz').dataobj[:]
            submask = np.multiply(submask, mask)

            # normalized BOLD time series of all voxels corresponding to each of the cortical & subcortical parcels
            epi = load_img('./data/brain/derivatives/'+subname+'/'+subname+'_task-'+task+'_smoothed.nii.gz').dataobj[:]
            roi_ts = {}
            for roi in range(1, nroi_cor+nroi_sub+1):
                mask_id = np.where(submask == roi)
                ts = np.array([])
                ts = epi[mask_id[0], mask_id[1], mask_id[2], :]

                # remove time steps that were censored due to motion
                for tr in range(ts.shape[1]):
                    if np.all(ts[:,tr]==0):
                        ts[:,tr] = np.nan
                # normalize each voxel time series
                ts = scipy.stats.zscore(ts, axis=1, ddof=1, nan_policy='omit')
                # load all ROIs in one run of one subject
                roi_ts[roi] = ts
            # load all 10 runs of one subject
            roi_ts_run[ti] = roi_ts
        # load all subjects in a group
        allsub_ROIsum[si] = roi_ts_run
        file_name = "./data/brain/loaded_BOLD/ROIsum_combined_mask_g"+str(groupid)+".pkl"
        with open(file_name, "wb") as file:
            pickle.dump(allsub_ROIsum, file)

Loading data from sub-1001
sub-1001 task-01
sub-1001 task-02
sub-1001 task-03
sub-1001 task-04
sub-1001 task-05
sub-1001 task-06
sub-1001 task-07
sub-1001 task-08
sub-1001 task-09
sub-1001 task-10
Loading data from sub-1005
sub-1005 task-01
sub-1005 task-02
sub-1005 task-03
sub-1005 task-04
sub-1005 task-05
sub-1005 task-06
sub-1005 task-07
sub-1005 task-08
sub-1005 task-09
sub-1005 task-10
Loading data from sub-1008
sub-1008 task-01
sub-1008 task-02
sub-1008 task-03
sub-1008 task-04
sub-1008 task-05
sub-1008 task-06
sub-1008 task-07
sub-1008 task-08
sub-1008 task-09
sub-1008 task-10
Loading data from sub-1011
sub-1011 task-01
sub-1011 task-02
sub-1011 task-03
sub-1011 task-04
sub-1011 task-05
sub-1011 task-06
sub-1011 task-07
sub-1011 task-08
sub-1011 task-09
sub-1011 task-10
Loading data from sub-1014
sub-1014 task-01
sub-1014 task-02
sub-1014 task-03
sub-1014 task-04
sub-1014 task-05
sub-1014 task-06
sub-1014 task-07
sub-1014 task-08
sub-1014 task-09
sub-1014 task-10
Loading data fr

# subselect the movie-watching part of the brain data

In [3]:
def load_events(subname,task):
    tst = pd.read_csv('./data/brain/events/'+subname+'_task-'+task+'_events.tsv', sep='\t')
    tst['offset'] = tst['onset'] + tst['duration']
    tst['onset'] = tst['onset'] + hrf
    tst['offset'] = tst['offset'] + hrf
    if task == '07':
        onset = tst['onset'][:3]
        offset = tst['offset'][:3]
    else:
        onset = tst['onset'][:5]
        offset = tst['offset'][:5]
    return onset, offset

In [4]:
for groupid in range(1,3+1):
    print('running group',str(groupid))
    file_name = "./data/brain/loaded_BOLD/ROIsum_combined_mask_g"+str(groupid)+".pkl"
    with open(file_name, "rb") as file:
        loaded_data = pickle.load(file)
    
    roi_pattern = {}
    # loop through ROIs to calculate between-participant similarity
    for roi in range(1,nroi+1):
        print('Running ROI',str(roi),'/116')
    
        # loop through subjects
        for si in range(len(flist[groupid])):
            # print('Running Sub',str(si+1),'/',str(len(flist[groupid])))
            # loop through tasks:
            for ti, task in enumerate(tasklist):
                brain = loaded_data[si][ti][roi]
                onset,offset = load_events(flist[groupid][si],task)
                # loop through the movie charcters
                events = []
                for k in range(len(onset)):
                    eventk = brain[:,int(onset[k]):int(offset[k])] # load the voxel time series
                    events.append(np.array(eventk))
                roi_pattern[roi,si,ti] = events
    
    savename = "./data/brain/loaded_BOLD/encoding_byevent_g"+str(groupid)+".pkl"
    with open(savename, "wb") as file:
        pickle.dump(roi_pattern, file)

running group 1
Running ROI 1 /116
Running ROI 2 /116
Running ROI 3 /116
Running ROI 4 /116
Running ROI 5 /116
Running ROI 6 /116
Running ROI 7 /116
Running ROI 8 /116
Running ROI 9 /116
Running ROI 10 /116
Running ROI 11 /116
Running ROI 12 /116
Running ROI 13 /116
Running ROI 14 /116
Running ROI 15 /116
Running ROI 16 /116
Running ROI 17 /116
Running ROI 18 /116
Running ROI 19 /116
Running ROI 20 /116
Running ROI 21 /116
Running ROI 22 /116
Running ROI 23 /116
Running ROI 24 /116
Running ROI 25 /116
Running ROI 26 /116
Running ROI 27 /116
Running ROI 28 /116
Running ROI 29 /116
Running ROI 30 /116
Running ROI 31 /116
Running ROI 32 /116
Running ROI 33 /116
Running ROI 34 /116
Running ROI 35 /116
Running ROI 36 /116
Running ROI 37 /116
Running ROI 38 /116
Running ROI 39 /116
Running ROI 40 /116
Running ROI 41 /116
Running ROI 42 /116
Running ROI 43 /116
Running ROI 44 /116
Running ROI 45 /116
Running ROI 46 /116
Running ROI 47 /116
Running ROI 48 /116
Running ROI 49 /116
Running ROI 5

# subselect the talking part of the brain data

In [5]:
def load_chars_impression(subname,task):
    tst = pd.read_csv('./data/brain/events/'+subname+'_task-'+task+'_events.tsv', sep='\t')
    tst['offset'] = tst['onset'] + tst['duration']
    tst['onset'] = tst['onset'] + hrf
    tst['offset'] = tst['offset'] + hrf
    onset = tst['onset'][-4:]
    offset = tst['offset'][-4:]
    return onset.tolist(), offset.tolist()

In [6]:
for groupid in range(1,3+1):
    print('running group',str(groupid))
    file_name = "./data/brain/loaded_BOLD/ROIsum_combined_mask_g"+str(groupid)+".pkl"
    with open(file_name, "rb") as file:
        loaded_data = pickle.load(file)
    
    roi_pattern = {}
    # loop through ROIs to calculate between-participant similarity
    for roi in range(1,nroi+1):
        print('Running ROI',str(roi),'/116')
    
        # loop through subjects
        for si in range(len(flist[groupid])):
            # print('Running Sub',str(si+1),'/',str(len(flist[groupid])))
            # loop through tasks:
            for ti, task in enumerate(tasklist):
                brain = loaded_data[si][ti][roi]
                onset,offset = load_chars_impression(flist[groupid][si],task)
                # loop through the movie charcters
                events = []
                for k in range(len(onset)):
                    eventk = brain[:,int(onset[k]):int(offset[k])] # load the voxel time series
                    events.append(np.array(eventk))
                roi_pattern[roi,si,ti] = events
    
    savename = "./data/brain/loaded_BOLD/impression_bychar_g"+str(groupid)+".pkl"
    with open(savename, "wb") as file:
        pickle.dump(roi_pattern, file)

running group 1
Running ROI 1 /116
Running ROI 2 /116
Running ROI 3 /116
Running ROI 4 /116
Running ROI 5 /116
Running ROI 6 /116
Running ROI 7 /116
Running ROI 8 /116
Running ROI 9 /116
Running ROI 10 /116
Running ROI 11 /116
Running ROI 12 /116
Running ROI 13 /116
Running ROI 14 /116
Running ROI 15 /116
Running ROI 16 /116
Running ROI 17 /116
Running ROI 18 /116
Running ROI 19 /116
Running ROI 20 /116
Running ROI 21 /116
Running ROI 22 /116
Running ROI 23 /116
Running ROI 24 /116
Running ROI 25 /116
Running ROI 26 /116
Running ROI 27 /116
Running ROI 28 /116
Running ROI 29 /116
Running ROI 30 /116
Running ROI 31 /116
Running ROI 32 /116
Running ROI 33 /116
Running ROI 34 /116
Running ROI 35 /116
Running ROI 36 /116
Running ROI 37 /116
Running ROI 38 /116
Running ROI 39 /116
Running ROI 40 /116
Running ROI 41 /116
Running ROI 42 /116
Running ROI 43 /116
Running ROI 44 /116
Running ROI 45 /116
Running ROI 46 /116
Running ROI 47 /116
Running ROI 48 /116
Running ROI 49 /116
Running ROI 5